# Week 3 — live-coding notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smc77/uc_finmlai/blob/main/lectures/week03/week3_demos.ipynb)

At the first break, run any **Setup** cell and then the **Imports** cell. If there is no Setup cell, begin with Imports. After that, use the slide cue to jump to the named demo; you do not need to rerun the whole notebook at every break. To check the whole notebook from a clean start, use **Runtime → Run all**.

Each demo follows the same rhythm: predict what the output should show, run the cell, and follow the task immediately underneath it. When an editable research-record entry appears, its completed comparison sits below it in a collapsed box—write your version before opening that box. This notebook is generated from the same source code the lecturer runs live.

## Jump to a demonstration

Use **Jupyter** when this notebook is open locally. Use **Colab** to open the hosted notebook directly at that demonstration. You do not need to scroll through or rerun the entire notebook.

### Deck A

<ul>
<li>Demo 1 — Fit OLS and read the coefficients with units — <a href="#Demo-1-%E2%80%94-Fit-OLS-and-read-the-coefficients-with-units">Jupyter</a> · <a href="https://colab.research.google.com/github/smc77/uc_finmlai/blob/main/lectures/week03/week3_demos.ipynb#scrollTo=md-10c507e2f5c6">Colab</a></li>
<li>Real-data companion — OLS explanation and OLS prediction are different jobs — <a href="#Real-data-companion-%E2%80%94-OLS-explanation-and-OLS-prediction-are-different-jobs">Jupyter</a> · <a href="https://colab.research.google.com/github/smc77/uc_finmlai/blob/main/lectures/week03/week3_demos.ipynb#scrollTo=md-c4f649dc7be9">Colab</a></li>
<li>Demo 2 — Selection vs shrinkage on one wide design — <a href="#Demo-2-%E2%80%94-Selection-vs-shrinkage-on-one-wide-design">Jupyter</a> · <a href="https://colab.research.google.com/github/smc77/uc_finmlai/blob/main/lectures/week03/week3_demos.ipynb#scrollTo=md-d65ce70b3d58">Colab</a></li>
<li>Demo 3 — Robust loss and quantiles answer different questions — <a href="#Demo-3-%E2%80%94-Robust-loss-and-quantiles-answer-different-questions">Jupyter</a> · <a href="https://colab.research.google.com/github/smc77/uc_finmlai/blob/main/lectures/week03/week3_demos.ipynb#scrollTo=md-906d2e09d045">Colab</a></li>
</ul>

### Deck B

<ul>
<li>Demo 4 — Baseline and predictive R² with a sample rank IC — <a href="#Demo-4-%E2%80%94-Baseline-and-predictive-R%C2%B2-with-a-sample-rank-IC">Jupyter</a> · <a href="https://colab.research.google.com/github/smc77/uc_finmlai/blob/main/lectures/week03/week3_demos.ipynb#scrollTo=md-2b9dbcfdcb8d">Colab</a></li>
<li>Demo 5 — Metrics: scale, ordering, and outlier sensitivity — <a href="#Demo-5-%E2%80%94-Metrics%3A-scale,-ordering,-and-outlier-sensitivity">Jupyter</a> · <a href="https://colab.research.google.com/github/smc77/uc_finmlai/blob/main/lectures/week03/week3_demos.ipynb#scrollTo=md-1b62a1cc1dd3">Colab</a></li>
<li>Demo 6 — Forecast to position, return, turnover, and cost — <a href="#Demo-6-%E2%80%94-Forecast-to-position,-return,-turnover,-and-cost">Jupyter</a> · <a href="https://colab.research.google.com/github/smc77/uc_finmlai/blob/main/lectures/week03/week3_demos.ipynb#scrollTo=md-bc8fb5b072f2">Colab</a></li>
</ul>

## Your Week 3 practice research record

Complete the editable entry immediately below each demonstration. **Do not write in this overview.** The six local entries form your Week 3 practice record. They remain in this notebook and are not submitted separately; the final-project record begins in Week 6.

### The six lecture breaks

1. **Fit — Demo 1:** record the target clock, feature and target units, fit dates, preprocessing, coefficients, and the limit of a training fit.
2. **Select — Demo 2:** record the search breadth, three-way split, validation choices, and untouched test comparison.
3. **Estimate — Demo 3:** keep the robust-loss and quantile experiments separate; record their targets, fitted slopes, and common controls.
4. **Compete — Demo 4:** name the feasible baseline, fit cutoff, test dates, both losses, predictive R², and subperiod stability.
5. **Score — Demo 5:** distinguish magnitude from ordering on one frozen assessment sample and declare which metric fits the intended use.
6. **Trade — Demo 6:** record the decision clock, position rule, first-position convention, cost units, turnover, gross, net, and reconciliation.

The real-data companion after Demo 1 is optional and has its own comparison entry. For every result, state one narrow claim and one limitation.

In [ ]:
# Setup: install packages not preinstalled in Colab.
%pip install -q finmlsim

### Imports

> **Run this once before any demo.** It loads the packages and helper functions used below; there is no result to interpret in this cell.

In [ ]:
from io import BytesIO, StringIO
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import (LinearRegression, Ridge, Lasso, ElasticNet,
                                  HuberRegressor, QuantileRegressor)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from finmlsim.simulate import garch

def course_csv(relative_path):
    """Read bundled Fama-French data locally, or its public source in Colab."""
    for root in (Path.cwd(), *Path.cwd().parents):
        local = root / relative_path
        if local.exists():
            return pd.read_csv(local), str(local)
    archives = {
        "datasets/famafrench/ff_factors_daily.csv": "F-F_Research_Data_Factors_daily_CSV.zip",
        "datasets/famafrench/ff_12industry_daily.csv": "12_Industry_Portfolios_daily_CSV.zip",
    }
    archive = archives[relative_path]
    url = f"https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/{archive}"
    with urlopen(url) as response, ZipFile(BytesIO(response.read())) as zipped:
        lines = zipped.read(zipped.namelist()[0]).decode("utf-8").splitlines()
    header_row = next(i for i, line in enumerate(lines) if line.startswith(","))
    rows = []
    for line in lines[header_row + 1:]:
        first = line.split(",", 1)[0].strip()
        if len(first) == 8 and first.isdigit():
            rows.append(line)
        elif rows:
            break
    frame = pd.read_csv(StringIO("\n".join([lines[header_row], *rows])))
    return frame.rename(columns={frame.columns[0]: "date"}), url


def build_week3_forecast_row():
    """Return the deterministic Week 3 row used by Demos 1 and 4.

    GARCH supplies changing conditional variance but no directional conditional
    mean.  Consequently, any tiny positive forecasting score is sampling noise,
    not a planted return edge.
    """
    n_obs = 2000
    dates = pd.bdate_range("2017-01-02", periods=n_obs)
    returns = pd.Series(
        garch(n=n_obs, omega=1e-6, alpha=0.07, beta=0.90, seed=1),
        index=dates,
        name="return",
    )
    features = pd.DataFrame({
        "mom21": returns.rolling(21).mean().shift(1),
        "vol21": returns.rolling(21).std().shift(1),
        "mom5": returns.rolling(5).mean().shift(1),
    })
    target = returns.shift(-1).rename("next_return")
    row = pd.concat([features, target], axis=1).dropna()
    split_at = int(0.7 * len(row))
    return returns, features, row, row.iloc[:split_at], row.iloc[split_at:]

### Demo 1 — Fit OLS and read the coefficients with units

> **Break cue:** Deck A, after recording segment S1. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

In [ ]:
# A known-truth row: volatility changes, but expected next-session return is zero.
ret, X, data, train, test = build_week3_forecast_row()
X_cols = list(X.columns)

pipe = Pipeline([("scale", StandardScaler()), ("ols", LinearRegression())])
pipe.fit(train[X_cols], train["next_return"])
ols_step = pipe.named_steps["ols"]
print("OLS coefficients (per 1 SD of each feature; target = next-day log return):")
for name, c in zip(X_cols, ols_step.coef_):
    print(f"  {name:8s}: {c:+.6f}")
print(f"  intercept: {ols_step.intercept_:+.6f}")
print(f"Fit rows:  {train.index.min().date()} through {train.index.max().date()} ({len(train)} rows)")
print(f"Test rows: {test.index.min().date()} through {test.index.max().date()} ({len(test)} rows)")
print("Clock: features end at the prior close; target is the next session's return.")
print("A coefficient is meaningless without its feature and target units.")
# Expected: coefficients are tiny. This known-truth process has volatility memory
#           but no planted directional edge, and training fit is not a forward claim.

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| target and clock | `[outcome, unit, and forecast interval]` |
| features and units | `[each feature, its unit, and availability cutoff]` |
| fit boundary | `[fit dates/rows and where assessment begins]` |
| preprocessing and model | `[ordered fitted steps]` |
| fitted coefficients | `[coefficient beside its feature and target units]` |
| narrow claim | `[what the fitted table establishes]` |
| limitation | `[what training fit cannot establish]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed fit entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| target and clock | next-session log return in decimal units; features stop at the prior close |
| features and units | `mom21` and `mom5`: average daily log return; `vol21`: daily-return standard deviation; each is standardized using training rows only |
| fit boundary | 1,384 rows, 2017-01-31 through 2022-05-20; later assessment rows begin 2022-05-23 |
| preprocessing and model | `StandardScaler` followed by OLS inside one pipeline |
| fitted coefficients | per one training-sample feature SD: `mom21` **−0.000052**, `vol21` **−0.000087**, `mom5` **−0.000019**; intercept **+0.000052** |
| narrow claim | the fitted line is legible because every coefficient has a feature unit, target unit, and fit boundary |
| limitation | this is a training fit in a known-truth process with no directional edge; it is not evidence that the features forecast later returns |

The small coefficients are not the conclusion. The complete measurement and fit
description is what makes them interpretable.

</details>

### Real-data companion — OLS explanation and OLS prediction are different jobs

> **Optional clock-and-target comparison:** Deck A, after recording segment S1. Run this after Demo 1. The two regressions deliberately answer different questions: one explains a same-day industry return; the other forecasts the next market return. Record both clocks and targets before comparing their R² values.

In [ ]:
ff_factors_raw, ff_factor_source = course_csv("datasets/famafrench/ff_factors_daily.csv")
ff_industry_raw, ff_industry_source = course_csv("datasets/famafrench/ff_12industry_daily.csv")
for frame in (ff_factors_raw, ff_industry_raw):
    frame["date"] = pd.to_datetime(frame["date"].astype(str), format="%Y%m%d")
ff_joined = ff_factors_raw.merge(ff_industry_raw, on="date").set_index("date").sort_index()
ff_joined = ff_joined.loc["1990":]

# Contemporaneous explanatory regression, in percentage points.
factor_columns = ["Mkt-RF", "SMB", "HML"]
industry_excess = ff_joined["BusEq"] - ff_joined["RF"]
real_cut = int(0.7 * len(ff_joined))
factor_ols = LinearRegression().fit(ff_joined[factor_columns].iloc[:real_cut], industry_excess.iloc[:real_cut])
factor_prediction = factor_ols.predict(ff_joined[factor_columns].iloc[real_cut:])
factor_truth = industry_excess.iloc[real_cut:].to_numpy()
factor_r2 = 1 - np.sum((factor_truth - factor_prediction) ** 2) / np.sum(
    (factor_truth - industry_excess.iloc[:real_cut].mean()) ** 2
)

# Forward prediction uses only information available after close t.
market_real = (ff_joined["Mkt-RF"] + ff_joined["RF"]) / 100.0
forecast_frame = pd.DataFrame({
    "return_t": market_real,
    "mom5": market_real.rolling(5).mean(),
    "mom21": market_real.rolling(21).mean(),
    "vol21": market_real.rolling(21).std(),
    "next_return": market_real.shift(-1),
}).dropna()
forecast_cut = int(0.7 * len(forecast_frame))
forecast_train = forecast_frame.iloc[:forecast_cut]
forecast_test = forecast_frame.iloc[forecast_cut:]
forecast_columns = ["return_t", "mom5", "mom21", "vol21"]
forecast_ols = Pipeline([("scale", StandardScaler()), ("ols", LinearRegression())])
forecast_ols.fit(forecast_train[forecast_columns], forecast_train["next_return"])
forecast_prediction = forecast_ols.predict(forecast_test[forecast_columns])
forecast_r2 = 1 - np.sum((forecast_test["next_return"] - forecast_prediction) ** 2) / np.sum(
    (forecast_test["next_return"] - forecast_train["next_return"].mean()) ** 2
)

print("Source: Kenneth R. French Data Library, bundled factors and 12 industries")
print(f"Files: {ff_factor_source}; {ff_industry_source}")
print(f"Sample: {ff_joined.index.min().date()} through {ff_joined.index.max().date()}")
print("Units: factor regression in percentage points; forecast regression in decimal daily returns")
print("Forecast clock: decide after close t using data through close t; target is return t+1.")
print(f"Contemporaneous BusEq factor-model test R²: {factor_r2:+.3f}")
print(f"Next-day market forecast test R²:          {forecast_r2:+.4f}")
print("OLS can explain a strong contemporaneous relationship and still have little to forecast tomorrow.")

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| explanatory question | `[same-period target and inputs]` |
| forecasting question | `[future target and available inputs]` |
| sample and units | `[source, dates, and units for both regressions]` |
| held-out results | `[both R² values]` |
| narrow conclusion | `[what the contrast establishes]` |
| limitation | `[why these are not interchangeable contests]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed optional comparison**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| explanatory question | use same-day market, size, and value factors to explain Business Equipment industry excess returns |
| forecasting question | use information through close *t* to forecast the US market return on *t*+1 |
| sample and units | Kenneth R. French daily data, 1990-01-02 through 2026-04-30; factor regression in percentage points and forecast regression in decimal returns |
| held-out results | contemporaneous factor-model test R² **+0.888**; next-day market predictive R² **+0.0029** |
| narrow conclusion | the same model family can explain same-period variation and still forecast very little across a time boundary |
| limitation | this compares two different targets and information clocks; it is not a horse race between two interchangeable regressions |

The clock and target determine the claim. The word *OLS* does not.

</details>

### Demo 2 — Selection vs shrinkage on one wide design

> **Break cue:** Deck A, after recording segment S2. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

In [ ]:
# A few true features hide among many noise columns. Training fits candidates,
# validation chooses complexity, and a genuinely untouched test set assesses it.
rng = np.random.default_rng(2)
n, p, n_true = 450, 30, 4
Xw = rng.standard_normal((n, p))
beta = np.zeros(p); beta[:n_true] = [1.0, -0.8, 0.6, -0.5]
yw = Xw @ beta + rng.standard_normal(n) * 2.0
ntr, nval = 180, 120
Xtr, Xval, Xte = Xw[:ntr], Xw[ntr:ntr + nval], Xw[ntr + nval:]
ytr, yval, yte = yw[:ntr], yw[ntr:ntr + nval], yw[ntr + nval:]

# forward stepwise: validation MSE vs subset size
chosen, remaining, tr_mse, va_mse = [], list(range(p)), [], []
mu = ytr.mean()
tr_mse.append(np.mean((ytr - mu) ** 2)); va_mse.append(np.mean((yval - mu) ** 2))
for _ in range(min(p, 15)):
    best_j, best = None, np.inf
    for j in remaining:
        cols = chosen + [j]
        A = np.c_[np.ones(ntr), Xtr[:, cols]]
        b = np.linalg.lstsq(A, ytr, rcond=None)[0]
        m = np.mean((ytr - A @ b) ** 2)
        if m < best:
            best, best_j = m, j
    chosen.append(best_j); remaining.remove(best_j)
    A = np.c_[np.ones(ntr), Xtr[:, chosen]]
    b = np.linalg.lstsq(A, ytr, rcond=None)[0]
    tr_mse.append(np.mean((ytr - A @ b) ** 2))
    Aval = np.c_[np.ones(len(yval)), Xval[:, chosen]]
    va_mse.append(np.mean((yval - Aval @ b) ** 2))
k_star = int(np.argmin(va_mse))
selected_columns = chosen[:k_star]
stepwise_final = LinearRegression().fit(Xw[:ntr + nval, selected_columns],
                                        yw[:ntr + nval])
stepwise_test_mse = np.mean(
    (yte - stepwise_final.predict(Xte[:, selected_columns])) ** 2
)
candidate_additions = sum(p - k for k in range(15))
print("Split: train rows 0-179; validation rows 180-299; test rows 300-449.")
print(f"Forward stepwise considered {candidate_additions} candidate additions across 15 nested sizes.")
print(f"Validation selected size {k_star} (true = {n_true}); selected columns {selected_columns}.")
print("The test set was not used to select the path or its size.")

# Shrinkage: validation chooses alpha, then train+validation refit once for test.
specifications = [
    ("Ridge", lambda a: Ridge(alpha=a), [0.1, 1.0, 10.0, 30.0, 100.0, 300.0]),
    ("Lasso", lambda a: Lasso(alpha=a, max_iter=20_000), [0.01, 0.03, 0.1, 0.3, 1.0]),
    ("ElasticNet", lambda a: ElasticNet(alpha=a, l1_ratio=0.5, max_iter=20_000),
     [0.01, 0.03, 0.1, 0.3, 1.0]),
]
results = [{"model": "Stepwise", "selected": f"size={k_star}",
            "test_mse": stepwise_test_mse, "nonzero": k_star}]
ols_all = Pipeline([("scale", StandardScaler()), ("model", LinearRegression())])
ols_all.fit(Xw[:ntr + nval], yw[:ntr + nval])
results.append({"model": "OLS (all p)", "selected": "none",
                "test_mse": np.mean((yte - ols_all.predict(Xte)) ** 2), "nonzero": p})
for name, constructor, alpha_grid in specifications:
    validation_mse = []
    for alpha in alpha_grid:
        candidate = Pipeline([
            ("scale", StandardScaler()), ("model", constructor(alpha))
        ])
        candidate.fit(Xtr, ytr)
        validation_mse.append(np.mean((yval - candidate.predict(Xval)) ** 2))
    chosen_alpha = alpha_grid[int(np.argmin(validation_mse))]
    final = Pipeline([
        ("scale", StandardScaler()), ("model", constructor(chosen_alpha))
    ])
    final.fit(Xw[:ntr + nval], yw[:ntr + nval])
    coef = final.named_steps["model"].coef_
    results.append({"model": name, "selected": f"alpha={chosen_alpha:g}",
                    "test_mse": np.mean((yte - final.predict(Xte)) ** 2),
                    "nonzero": int(np.sum(np.abs(coef) > 1e-6))})
result_table = pd.DataFrame(results)
print("Final comparison on untouched test rows:")
print(result_table.to_string(index=False, formatters={"test_mse": "{:.3f}".format}))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(range(len(tr_mse)), tr_mse, "o-", label="training MSE")
ax.plot(range(len(va_mse)), va_mse, "s-", label="validation MSE")
ax.axvline(k_star, color="firebrick", ls="--")
ax.set_xlabel("features entered (forward stepwise)"); ax.set_ylabel("MSE")
ax.legend(); ax.set_title("Selection path: validation error turns")
fig.tight_layout(); plt.show()
# Expected: validation MSE turns at size 4, the true sparsity. Lasso and ElasticNet
#           reach lower test MSE than OLS with roughly half the nonzero coefficients.

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| known truth and candidates | `[rows, candidate columns, and planted signal count]` |
| split | `[training, validation, and untouched test rows]` |
| search breadth | `[candidate additions, subset sizes, and penalty grids]` |
| validation choices | `[selected size and each selected penalty]` |
| untouched test results | `[test MSE for every assessed procedure]` |
| selected complexity | `[nonzero coefficients or selected subset size]` |
| narrow claim | `[what this test comparison supports]` |
| limitation | `[what this design and split cannot establish]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed selection entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| known truth and candidates | 450 independent synthetic rows, 30 candidate features, four planted nonzero coefficients |
| split | training rows 0–179; validation rows 180–299; untouched test rows 300–449 |
| search breadth | forward stepwise evaluated 345 candidate additions across 15 nested subset sizes; ridge used six alpha values; lasso and elastic net used five each |
| validation choices | stepwise size **4**, ridge alpha **100**, lasso alpha **0.3**, elastic-net alpha **0.3** |
| untouched test results | MSE: elastic net **4.648**, stepwise **4.656**, lasso **4.658**, ridge **4.997**, all-feature OLS **5.112** |
| selected complexity | nonzero coefficients: stepwise 4, lasso 4, elastic net 6, ridge 30, OLS 30 |
| narrow claim | in this design, validation recovered the planted dimension and constrained procedures beat all-feature OLS on untouched test rows |
| limitation | one independent-row simulation and one split do not establish a universal winner or represent time-series validation |

The test set judges the complete selection procedure only after validation has
made every complexity choice.

</details>

### Demo 3 — Robust loss and quantiles answer different questions

> **Break cue:** Deck A, after recording segment S3. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

In [ ]:
# Two known-truth experiments isolate two different questions: robust loss for
# extreme residuals, then quantiles for a conditional distribution that fans out.
rng = np.random.default_rng(7)
nq = 200
xq = rng.standard_normal(nq)
yq = 0.9 * xq + rng.standard_normal(nq) * 0.4
yq[[5, 50, 120]] += [8.0, -7.0, 6.0]               # fat-tailed shocks
Xq = xq.reshape(-1, 1)
ols_slope = LinearRegression().fit(Xq, yq).coef_[0]
huber_slope = HuberRegressor().fit(Xq, yq).coef_[0]
print("Slope under fat-tailed shocks (true = 0.90):")
print(f"  OLS:   {ols_slope:+.2f}   (dragged by the shocks)")
print(f"  Huber: {huber_slope:+.2f}   (resists them)")

xh = np.sort(rng.uniform(0, 3, 400))
yh = 0.4 * xh + rng.standard_normal(400) * (0.25 + 0.5 * xh)   # spread grows with x
Xh = xh.reshape(-1, 1)
print("Quantile regression under heteroskedasticity (slope per quantile):")
quantile_rows = []
for tau in [0.1, 0.5, 0.9]:
    model = QuantileRegressor(quantile=tau, alpha=0.0, solver="highs").fit(Xh, yh)
    quantile_rows.append((tau, model.coef_[0], model.predict([[2.5]])[0]))
    print(f"  tau = {tau}: slope = {model.coef_[0]:+.2f}; forecast at x=2.5 = {model.predict([[2.5]])[0]:+.2f}")
spread_at_25 = quantile_rows[-1][2] - quantile_rows[0][2]
print(f"At x=2.5, the fitted 90th-minus-10th quantile spread is {spread_at_25:.2f}.")
print("Different quantiles are different targets; their distance describes conditional spread.")
# Expected: OLS is about 0.83 against a true 0.90 while Huber is about 0.92.
# Quantile slopes fan from about -0.17 to +0.96 in the separate variance experiment.

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| robust-loss experiment | `[known truth, rows, and planted problem]` |
| robust-loss evidence | `[OLS and Huber slopes]` |
| quantile experiment | `[separate distributional design]` |
| quantile evidence | `[levels, slopes, and forecasts at x=2.5]` |
| conditional spread | `[90th-minus-10th quantile distance]` |
| what stayed fixed | `[controls within each experiment]` |
| narrow claim | `[what changed when the loss or target changed]` |
| limitation | `[what these simulations cannot establish]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed estimator entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| robust-loss experiment | 200 synthetic rows with true central slope 0.90 and three planted extreme residuals |
| robust-loss evidence | OLS slope **+0.83**; Huber slope **+0.92** |
| quantile experiment | a separate 400-row heteroskedastic design whose conditional spread grows with `x` |
| quantile evidence | slopes at τ=0.1, 0.5, and 0.9: **−0.17**, **+0.35**, **+0.96**; forecasts at `x=2.5`: **−0.73**, **+0.88**, **+2.71** |
| conditional spread | fitted 90th-minus-10th quantile distance at `x=2.5`: **3.44** |
| what stayed fixed | within each experiment, rows and units stay fixed while only the loss or quantile target changes |
| narrow claim | Huber limits the pull of the planted extremes; different quantiles describe different parts of a widening conditional distribution |
| limitation | these are two controlled demonstrations, not evidence that either method improves a real return forecast |

Do not compare all five slopes as estimates of one quantity. The robust and
quantile demonstrations answer different questions.

</details>

### Demo 4 — Baseline and predictive R² with a sample rank IC

> **Break cue:** Deck B, after recording segment S4. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

In [ ]:
# Rebuild the row so this demo runs directly after Imports. Time-order split,
# train-only scaling, and a feasible baseline: the frozen training-period mean.
ret, X, data, train, test = build_week3_forecast_row()
X_cols = list(X.columns)
X_train, y_train = train[X_cols].values, train["next_return"].values
X_test, y_test = test[X_cols].values, test["next_return"].values

pipe = Pipeline([("scale", StandardScaler()), ("ols", LinearRegression())])
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
baseline_pred = np.repeat(y_train.mean(), len(y_test))
forecast_comparison = pd.DataFrame({
    "target": y_test,
    "forecast": y_pred,
    "baseline_forecast": baseline_pred,
}, index=test.index)

model_mse = np.mean((y_test - y_pred) ** 2)
baseline_mse = np.mean((y_test - baseline_pred) ** 2)
ss_res = np.sum((y_test - y_pred) ** 2)
ss_tot = np.sum((y_test - baseline_pred) ** 2)
predictive_r2 = 1 - ss_res / ss_tot
rank_ic, rank_ic_p = stats.spearmanr(y_pred, y_test)
print("Known truth: the simulation has volatility memory but zero directional conditional mean.")
print(f"Fit rows:  {train.index.min().date()} through {train.index.max().date()}")
print(f"Test rows: {test.index.min().date()} through {test.index.max().date()}")
print(f"Model MSE: {model_mse:.8f}; frozen train-mean baseline MSE: {baseline_mse:.8f}")
print("First five assessment rows:")
print(forecast_comparison.head().round(6).to_string())
print(f"Predictive R² (train-mean baseline): {predictive_r2:+.4f}   (often near 0)")
print(f"Test time-series Spearman:          {rank_ic:+.4f}   "
      f"(naive iid p={rank_ic_p:.3f}; not time-series inference)")
subperiod = [stats.spearmanr(y_pred[idx], y_test[idx]).statistic
             for idx in np.array_split(np.arange(len(y_test)), 4)]
print("Contiguous-quarter Spearman:        " + ", ".join(f"{v:+.4f}" for v in subperiod))
print("Narrow conclusion: this one held-out sample produced a near tie; it did not discover an edge.")

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| known truth | `[directional and volatility structure]` |
| model and feasible baseline | `[both forecast rules and update/freeze rule]` |
| fit and test boundaries | `[dates and common cutoff]` |
| loss evidence | `[model MSE, baseline MSE, and predictive R²]` |
| ordering and stability | `[full-period and contiguous-subperiod Spearman]` |
| narrow claim | `[what the held-out comparison supports]` |
| limitation | `[what the positive score and iid p-value cannot establish]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed baseline entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| known truth | the GARCH simulation has changing volatility but zero directional conditional mean |
| model and feasible baseline | training-fitted scaler plus OLS versus the training-period mean, frozen throughout assessment |
| fit and test boundaries | fit 2017-01-31–2022-05-20; test 2022-05-23–2024-08-29 |
| loss evidence | model MSE **0.00003631**; baseline MSE **0.00003633**; predictive R² **+0.0007** |
| ordering and stability | time-series Spearman **+0.0815**; contiguous-quarter values **−0.0009**, **+0.2050**, **+0.0779**, **+0.1145** |
| narrow claim | the fitted model and feasible baseline were nearly tied on this one held-out sample |
| limitation | because true directional predictability is zero, the tiny positive score is sampling noise—not discovery of an edge; the naive iid Spearman p-value is not time-series inference |

Known truth lets us say more than the score alone: a positive held-out number
can occur even when no directional edge exists.

</details>

### Demo 5 — Metrics: scale, ordering, and outlier sensitivity

> **Break cue:** Deck B, after recording segment S5. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

In [ ]:
# This is a frozen synthetic assessment sample: no model is selected or fitted here.
rng = np.random.default_rng(5)
n = 2000
metric_dates = pd.bdate_range("2017-01-02", periods=n)
true_signal = rng.standard_normal(n)
y_real = 0.3 * true_signal + rng.standard_normal(n)
base_forecast = 0.3 * true_signal

rows = []
for multiplier in [0.25, 1.0, 3.0]:
    pred = multiplier * base_forecast
    r2 = 1 - np.sum((y_real - pred) ** 2) / np.sum(y_real ** 2)
    rows.append({"multiplier": multiplier, "predictive_r2": r2,
                 "pearson": stats.pearsonr(pred, y_real).statistic,
                 "spearman": stats.spearmanr(pred, y_real).statistic})
print("Change forecast scale while preserving order:")
print("Predictive R² in this controlled metric exercise uses a zero forecast as its baseline.")
print(pd.DataFrame(rows).round(4).to_string(index=False))

grid = [0, 5, 20, 50, 100]
outlier_order = rng.permutation(n)
outlier_shock = np.zeros(n)
outlier_shock[outlier_order[:max(grid)]] = rng.standard_normal(max(grid)) * 20
outlier_rows = []
for k in grid:
    yp = y_real.copy()
    if k > 0:
        idx = outlier_order[:k]
        yp[idx] += outlier_shock[idx]
    outlier_rows.append({
        "outliers": k,
        "predictive_r2": 1 - np.sum((yp - base_forecast) ** 2) / np.sum(yp ** 2),
        "pearson": stats.pearsonr(base_forecast, yp).statistic,
        "spearman": stats.spearmanr(base_forecast, yp).statistic,
    })
outlier_table = pd.DataFrame(outlier_rows)
print(f"Frozen assessment rows: {metric_dates.min().date()} through {metric_dates.max().date()}")
print("Add a fixed, progressively larger set of outcome outliers:")
print(outlier_table.round(4).to_string(index=False))
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(grid, outlier_table["predictive_r2"], "o-", label="Predictive R²")
ax.plot(grid, outlier_table["pearson"], "^-", label="Pearson")
ax.plot(grid, outlier_table["spearman"], "s-", label="Spearman")
ax.set_xlabel("# outliers injected"); ax.set_ylabel("score")
ax.set_title("Ranks discard distance; squared loss does not")
ax.legend(); fig.tight_layout(); plt.show()
# Expected: rescaling moves predictive R2 but leaves both correlations unchanged.
# Forecast magnitude and ordering are different properties.

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| frozen assessment sample | `[rows, dates, and what was not fitted]` |
| scale experiment | `[multipliers applied to the fixed forecast]` |
| scale evidence | `[predictive R², Pearson, and Spearman table]` |
| outlier construction | `[fixed progressive contamination rule]` |
| outlier evidence | `[metric changes from zero to maximum contamination]` |
| intended use and primary metric | `[magnitude or ordering decision]` |
| invariance | `[transformation the chosen metric ignores]` |
| limitation | `[what the controlled exercise cannot establish]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed metric entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| frozen assessment sample | 2,000 synthetic rows dated 2017-01-02 through 2024-08-30; no fitting or metric selection occurs inside the exercise |
| scale experiment | multiply one fixed forecast by 0.25, 1, and 3 without changing its ordering; predictive R² uses a zero-forecast baseline |
| scale evidence | predictive R² **0.0353**, **0.0836**, **−0.2087**; Pearson stays **0.2888** and Spearman stays **0.2871** |
| outlier construction | add a fixed, progressively larger set of 5, 20, 50, then 100 extreme outcome shocks |
| outlier evidence | from 0 to 100 outliers: predictive R² falls **0.0836 → 0.0035**, Pearson **0.2888 → 0.0607**, Spearman **0.2871 → 0.2462** |
| intended use and primary metric | squared loss for a magnitude forecast; Spearman for a policy that uses only ordering |
| invariance | positive rescaling preserves ranks and both correlations but not squared forecast error |
| limitation | this controlled assessment sample demonstrates metric behavior, not profitability, model selection, or robustness on market data |

A metric is not a generic quality score. Its invariance must match what the
downstream decision actually uses.

</details>

### Demo 6 — Forecast to position, return, turnover, and cost

> **Break cue:** Deck B, after recording segment S6. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

In [ ]:
rng = np.random.default_rng(6)
n_dates = 1200
calendar = pd.bdate_range("2020-01-02", periods=n_dates + 1)
decision_dates, return_dates = calendar[:-1], calendar[1:]
score_feature = rng.standard_normal(n_dates)
next_return = 0.0008 * score_feature + 0.01 * rng.standard_normal(n_dates)
split_date = 800

policy = LinearRegression().fit(score_feature[:split_date].reshape(-1, 1),
                                next_return[:split_date])
train_forecast = policy.predict(score_feature[:split_date].reshape(-1, 1))
test_forecast = policy.predict(score_feature[split_date:].reshape(-1, 1))
test_return = next_return[split_date:]

position_scale = np.quantile(np.abs(train_forecast), 0.90)      # training-only
position = np.clip(test_forecast / position_scale, -1.0, 1.0)
previous = np.r_[0.0, position[:-1]]
turnover = np.abs(position - previous)
trading_cost = 0.0002 * turnover                               # 2 bps per unit
gross_return = position * test_return
net_return = gross_return - trading_cost

ledger = pd.DataFrame({
    "return_date": return_dates[split_date:],
    "forecast": test_forecast, "position": position,
    "realized_return": test_return, "gross_return": gross_return,
    "turnover": turnover, "trading_cost": trading_cost, "net_return": net_return,
}, index=pd.Index(decision_dates[split_date:], name="decision_date"))
print("Clock: decide after each decision date; hold the position for the next session's return.")
print(f"Model and position scale fitted through {decision_dates[split_date - 1].date()}.")
print(f"Training-only 90th percentile |forecast|: {position_scale:.6f}")
print("First position starts from cash; cost is 2 bps times absolute position change.")
print("\nFirst five rows of the single-series ledger:")
ledger_display = ledger.head().copy()
numeric_columns = ledger_display.select_dtypes(include="number").columns
ledger_display[numeric_columns] = ledger_display[numeric_columns].round(6)
print(ledger_display.to_string())
print(f"Total turnover:          {turnover.sum():.3f}")
print(f"Gross arithmetic return: {gross_return.sum():+.4f}")
print(f"Net arithmetic return:   {net_return.sum():+.4f}")
print("Reconciliation max error:",
      np.max(np.abs(gross_return - trading_cost - net_return)))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(np.cumsum(gross_return), label="gross")
ax.plot(np.cumsum(net_return), label="net of 2 bps × turnover")
ax.set_title("Forecast → position → return → turnover → cost")
ax.set_xlabel("held-out decision row"); ax.set_ylabel("cumulative return")
ax.legend(); fig.tight_layout(); plt.show()
# Expected: gross and net separate once turnover is priced. The ledger must reconcile
#           forecast -> position -> return -> cost.

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| decision and return clock | `[when the position is chosen and which return it earns]` |
| fit boundary | `[final fitting date and first assessment decision]` |
| position rule | `[forecast-to-position formula and training-only scale]` |
| first position and cost | `[starting exposure, turnover convention, and cost units]` |
| first five decision rows | `[copy or summarize the printed ledger rows]` |
| aggregate evidence | `[turnover, gross arithmetic return, and net arithmetic return]` |
| reconciliation | `[identity checked and maximum error]` |
| narrow claim | `[whether cost changed magnitude or sign]` |
| limitation | `[what this one policy cannot establish]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed trading-ledger entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| decision and return clock | decide after each dated row and hold the resulting position for the following session's return |
| fit boundary | model and position scale use rows through 2023-01-25 only; later decisions begin 2023-01-26 |
| position rule | divide the forecast by the training-only 90th percentile of absolute forecasts (**0.002312**) and clip exposure to [−1, +1] |
| first position and cost | begin from cash; turnover is absolute position change; charge 2 basis points per unit of turnover |
| first five decision rows | 2023-01-26 through 2023-02-01; the printed ledger records each next-session return date, forecast, position, return, turnover, cost, and net return |
| aggregate evidence | total turnover **224.277**; gross arithmetic return **+0.3371**; net arithmetic return **+0.2922** |
| reconciliation | maximum absolute error in `gross_return − trading_cost − net_return` is **0.0** |
| narrow claim | costs reduce the magnitude but not the sign of this one frozen synthetic policy's arithmetic result |
| limitation | one planted linear signal, one sizing rule, linear costs, and arithmetic aggregation do not establish a deployable strategy |

The ledger is the result because every net-return number can be traced backward
to a forecast, position, realized return, turnover, and cost.

</details>